# Etapa 5 — KD-Tree y KNN exacto con REES46

In [1]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent
DATA_PATH = PROJECT_DIR / "data" / "raw" / "2019-Oct.csv"

# Trabajaremos inicialmente con 100K registros
df_knn = pd.read_csv(
    DATA_PATH,
    nrows=100_000
)

# Contar eventos por producto y tipo
product_events = pd.crosstab(
    df_knn["product_id"],
    df_knn["event_type"]
)

# Garantizar que las tres columnas existan
for event_type in ["view", "cart", "purchase"]:
    if event_type not in product_events.columns:
        product_events[event_type] = 0

# Precio representativo: mediana observada
product_prices = df_knn.groupby(
    "product_id"
)["price"].median()

# Construir tabla de características
df_features = pd.DataFrame({
    "product_id": product_events.index,
    "price": product_prices.reindex(product_events.index).values,
    "views": product_events["view"].values,
    "carts": product_events["cart"].values,
    "purchases": product_events["purchase"].values
})

# Aplicar transformación logarítmica
df_features["log_price"] = np.log1p(
    df_features["price"].clip(lower=0)
)

for column in ["views", "carts", "purchases"]:
    df_features[f"log_{column}"] = np.log1p(
        df_features[column]
    )

print("=== FEATURES DE PRODUCTOS ===")
print("Productos:", len(df_features))
print("Columnas:", df_features.shape[1])

display(df_features.head(10))


=== FEATURES DE PRODUCTOS ===
Productos: 20621
Columnas: 9


,product_id,price,views,carts,purchases,log_price,log_views,log_carts,log_purchases
0,1001588,128.31,8,0,0,4.862213,2.197225,0.0,0.0
1,1002062,77.14,1,0,0,4.358502,0.693147,0.0,0.0
2,1002098,409.02,9,0,0,6.016206,2.302585,0.0,0.0
3,1002099,370.41,75,0,0,5.917307,4.330733,0.0,0.0
4,1002100,409.02,13,0,0,6.016206,2.639057,0.0,0.0
5,1002101,409.02,24,0,0,6.016206,3.218876,0.0,0.0
6,1002102,414.53,16,0,0,6.029555,2.833213,0.0,0.0
7,1002225,82.37,1,0,0,4.423289,0.693147,0.0,0.0
8,1002396,275.55,4,0,0,5.622392,1.609438,0.0,0.0
9,1002398,252.39,5,0,0,5.534930,1.791759,0.0,0.0


In [2]:

print("=== VALIDACIÓN DE FEATURES ===")

print("\nValores nulos:")
print(df_features.isna().sum())

print("\nEstadísticas descriptivas:")
display(
    df_features[
        ["log_price", "log_views", "log_carts", "log_purchases"]
    ].describe().round(3)
)

print("\nProductos con compras:")
print((df_features["purchases"] > 0).sum())

print("\nProductos con carritos:")
print((df_features["carts"] > 0).sum())


=== VALIDACIÓN DE FEATURES ===

Valores nulos:
product_id       0
price            0
views            0
carts            0
purchases        0
log_price        0
log_views        0
log_carts        0
log_purchases    0
dtype: int64

Estadísticas descriptivas:


,log_price,log_views,log_carts,log_purchases
count,20621.000,20621.000,20621.000,20621.000
mean,4.582,1.254,0.017,0.036
std,1.318,0.734,0.168,0.200
min,0.000,0.000,0.000,0.000
25%,3.766,0.693,0.000,0.000
50%,4.572,1.099,0.000,0.000
75%,5.476,1.609,0.000,0.000
max,7.854,6.903,4.431,4.127



Productos con compras:
843

Productos con carritos:
266


## 2. Normalización de características

In [3]:

from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

FEATURE_COLUMNS = [
    "log_price",
    "log_views",
    "log_carts",
    "log_purchases"
]

# Matriz original de características
X = df_features[FEATURE_COLUMNS].to_numpy(
    dtype=np.float64
)

# Normalización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Validaciones
assert np.isfinite(X_scaled).all()
assert X_scaled.shape == (20621, 4)

print("=== NORMALIZACIÓN ===")
print("Dimensiones:", X_scaled.shape)

print("\nMedia de cada característica:")
print(np.round(X_scaled.mean(axis=0), 6))

print("\nDesviación estándar:")
print(np.round(X_scaled.std(axis=0), 6))

print("\n=== PRIMEROS 5 VECTORES ===")
display(
    pd.DataFrame(
        X_scaled[:5],
        columns=FEATURE_COLUMNS
    ).round(4)
)


=== NORMALIZACIÓN ===
Dimensiones: (20621, 4)

Media de cada característica:
[-0. -0.  0. -0.]

Desviación estándar:
[1. 1. 1. 1.]

=== PRIMEROS 5 VECTORES ===


,log_price,log_views,log_carts,log_purchases
0,0.2124,1.2851,-0.0984,-0.1827
1,-0.1697,-0.7636,-0.0984,-0.1827
2,1.0879,1.4286,-0.0984,-0.1827
3,1.0129,4.1910,-0.0984,-0.1827
4,1.0879,1.8868,-0.0984,-0.1827


## 3. KD-Tree y búsqueda KNN exacta

In [4]:

from sklearn.neighbors import KDTree
import time

# Construir índice KD-Tree
inicio = time.perf_counter()

kd_tree = KDTree(
    X_scaled,
    leaf_size=40,
    metric="euclidean"
)

tiempo_construccion = time.perf_counter() - inicio

# Seleccionar un producto real para consultar
query_position = 0
query_vector = X_scaled[query_position:query_position + 1]

# Recuperar 6 vecinos, incluyendo el propio producto
distances, indices = kd_tree.query(
    query_vector,
    k=6
)

print("=== KD-TREE ===")
print(f"Productos indexados: {len(X_scaled):,}")
print(f"Tiempo de construcción: {tiempo_construccion:.4f} s")

print("\n=== PRODUCTO CONSULTADO ===")
print("Product ID:", df_features.iloc[query_position]["product_id"])

print("\n=== VECINOS MÁS CERCANOS ===")

for distance, index in zip(distances[0], indices[0]):
    product_id = df_features.iloc[index]["product_id"]

    print(
        f"Producto: {int(product_id)} | "
        f"Distancia: {distance:.6f}"
    )


=== KD-TREE ===
Productos indexados: 20,621
Tiempo de construcción: 0.0097 s

=== PRODUCTO CONSULTADO ===
Product ID: 1001588.0

=== VECINOS MÁS CERCANOS ===
Producto: 1001588 | Distancia: 0.000000
Producto: 28715802 | Distancia: 0.000821
Producto: 28715852 | Distancia: 0.000821
Producto: 14700810 | Distancia: 0.008318
Producto: 5000637 | Distancia: 0.013078
Producto: 3500243 | Distancia: 0.015735


## 4. Validación KD-Tree vs KNN exacto por fuerza bruta

In [5]:

from sklearn.neighbors import NearestNeighbors
import numpy as np
import time

# Modelo exacto por fuerza bruta
brute_model = NearestNeighbors(
    n_neighbors=6,
    algorithm="brute",
    metric="euclidean"
)

brute_model.fit(X_scaled)

# Consultar el mismo producto
query_vector = X_scaled[0:1]

# KD-Tree
inicio = time.perf_counter()
dist_kd, idx_kd = kd_tree.query(query_vector, k=6)
tiempo_kd = (time.perf_counter() - inicio) * 1000

# Fuerza bruta
inicio = time.perf_counter()
dist_brute, idx_brute = brute_model.kneighbors(
    query_vector
)
tiempo_brute = (time.perf_counter() - inicio) * 1000

# Comprobar las distancias ordenadas
distancias_coinciden = np.allclose(
    dist_kd[0],
    dist_brute[0],
    atol=1e-8
)

print("=== COMPARACIÓN KNN EXACTO ===")
print(f"Tiempo KD-Tree: {tiempo_kd:.4f} ms")
print(f"Tiempo fuerza bruta: {tiempo_brute:.4f} ms")

print("\nDistancias coinciden:", distancias_coinciden)

print("\n=== VECINOS KD-TREE ===")
print(idx_kd[0])

print("\n=== VECINOS FUERZA BRUTA ===")
print(idx_brute[0])

assert distancias_coinciden

print("\n¡KD-Tree validado contra búsqueda exacta!")


=== COMPARACIÓN KNN EXACTO ===
Tiempo KD-Tree: 0.4961 ms
Tiempo fuerza bruta: 37.8544 ms

Distancias coinciden: True

=== VECINOS KD-TREE ===
[    0 17795 17811 11449  4859  3179]

=== VECINOS FUERZA BRUTA ===
[    0 17811 17795 11449  4859  3179]

¡KD-Tree validado contra búsqueda exacta!


## 5. Benchmark KD-Tree vs fuerza bruta

In [6]:

import numpy as np
import pandas as pd
import time

rng = np.random.default_rng(42)

# Seleccionar 100 productos para consultar
query_positions = rng.choice(
    len(X_scaled),
    size=100,
    replace=False
)

query_vectors = X_scaled[query_positions]

K = 5

# Calentamiento de ambos métodos
kd_tree.query(query_vectors[:5], k=K + 1)
brute_model.kneighbors(query_vectors[:5], n_neighbors=K + 1)

# Ejecutar las 100 consultas como lote
inicio = time.perf_counter()
dist_kd, idx_kd = kd_tree.query(
    query_vectors,
    k=K + 1
)
tiempo_kd_ms = (time.perf_counter() - inicio) * 1000

inicio = time.perf_counter()
dist_brute, idx_brute = brute_model.kneighbors(
    query_vectors,
    n_neighbors=K + 1
)
tiempo_brute_ms = (time.perf_counter() - inicio) * 1000

# Verificar distancias de los primeros 6 vecinos
distancias_correctas = np.allclose(
    dist_kd,
    dist_brute,
    atol=1e-8
)

print("=== BENCHMARK KNN EXACTO ===")
print(f"Consultas: {len(query_vectors)}")
print(f"K: {K}")
print(f"Tiempo total KD-Tree: {tiempo_kd_ms:.4f} ms")
print(f"Tiempo total fuerza bruta: {tiempo_brute_ms:.4f} ms")

print("\n=== TIEMPO PROMEDIO ===")
print(f"KD-Tree: {tiempo_kd_ms / 100:.6f} ms/consulta")
print(f"Fuerza bruta: {tiempo_brute_ms / 100:.6f} ms/consulta")

print("\n=== VALIDACIÓN ===")
print("Distancias coinciden:", distancias_correctas)

assert distancias_correctas

print("\n¡Benchmark inicial completado!")


=== BENCHMARK KNN EXACTO ===
Consultas: 100
K: 5
Tiempo total KD-Tree: 3.1976 ms
Tiempo total fuerza bruta: 3.7258 ms

=== TIEMPO PROMEDIO ===
KD-Tree: 0.031976 ms/consulta
Fuerza bruta: 0.037258 ms/consulta

=== VALIDACIÓN ===
Distancias coinciden: False


AssertionError: 

## 6. Diagnóstico de diferencias entre KD-Tree y fuerza bruta

In [7]:

import numpy as np
import pandas as pd

# Diferencias absolutas entre distancias ordenadas
diff = np.abs(dist_kd - dist_brute)

# Identificar consultas con diferencias
tolerancia = 1e-8

consultas_diferentes = np.where(
    ~np.all(
        np.isclose(
            dist_kd,
            dist_brute,
            atol=tolerancia,
            rtol=1e-5
        ),
        axis=1
    )
)[0]

print("=== DIAGNÓSTICO KNN ===")
print("Consultas evaluadas:", len(query_vectors))
print("Consultas con diferencias:", len(consultas_diferentes))
print("Máxima diferencia absoluta:", diff.max())
print("Diferencia media absoluta:", diff.mean())

if len(consultas_diferentes) > 0:
    i = consultas_diferentes[0]

    print("\n=== PRIMERA CONSULTA CON DIFERENCIAS ===")
    print("Posición de consulta:", query_positions[i])
    print("Distancias KD-Tree:", dist_kd[i])
    print("Distancias fuerza bruta:", dist_brute[i])
    print("Diferencias absolutas:", diff[i])

    print("\nÍndices KD-Tree:", idx_kd[i])
    print("Índices fuerza bruta:", idx_brute[i])


=== DIAGNÓSTICO KNN ===
Consultas evaluadas: 100
Consultas con diferencias: 10
Máxima diferencia absoluta: 4.2146848510894035e-08
Diferencia media absoluta: 8.959474490646581e-10

=== PRIMERA CONSULTA CON DIFERENCIAS ===
Posición de consulta: 6707
Distancias KD-Tree: [0. 0. 0. 0. 0. 0.]
Distancias fuerza bruta: [2.98023224e-08 2.98023224e-08 2.98023224e-08 2.98023224e-08
 2.98023224e-08 2.98023224e-08]
Diferencias absolutas: [2.98023224e-08 2.98023224e-08 2.98023224e-08 2.98023224e-08
 2.98023224e-08 2.98023224e-08]

Índices KD-Tree: [6707 1154 6562  830  639 2317]
Índices fuerza bruta: [6707 6708  639  830 2317 1154]


## 7. Validación de exactitud con tolerancia numérica

In [8]:

import numpy as np

# Tolerancia numérica explícita
ABS_TOL = 1e-7

# Comparar distancias ordenadas
distancias_correctas = np.allclose(
    dist_kd,
    dist_brute,
    atol=ABS_TOL,
    rtol=0
)

# Calcular diferencia máxima
max_difference = np.max(
    np.abs(dist_kd - dist_brute)
)

# Número de consultas correctas
validaciones = np.all(
    np.isclose(
        dist_kd,
        dist_brute,
        atol=ABS_TOL,
        rtol=0
    ),
    axis=1
)

print("=== VALIDACIÓN KNN EXACTO ===")
print("Consultas evaluadas:", len(validaciones))
print("Consultas correctas:", validaciones.sum())

print(f"Diferencia máxima: {max_difference:.12f}")
print(f"Tolerancia utilizada: {ABS_TOL:.12f}")

print("\nDistancias equivalentes:", distancias_correctas)

assert distancias_correctas

print("\n¡KD-Tree validado correctamente!")


=== VALIDACIÓN KNN EXACTO ===
Consultas evaluadas: 100
Consultas correctas: 100
Diferencia máxima: 0.000000042147
Tolerancia utilizada: 0.000000100000

Distancias equivalentes: True

¡KD-Tree validado correctamente!


## 8. Evaluación de Recall@5 y empates

In [9]:

import numpy as np
import pandas as pd

K = 5
ABS_TOL = 1e-7

results = []

for i, query_id in enumerate(query_positions):

    # Vecinos de ambos métodos, excluyendo
    # el propio producto por su índice.
    kd_pairs = [
        (int(idx), float(dist))
        for idx, dist in zip(idx_kd[i], dist_kd[i])
        if idx != query_id
    ][:K]

    brute_pairs = [
        (int(idx), float(dist))
        for idx, dist in zip(idx_brute[i], dist_brute[i])
        if idx != query_id
    ][:K]

    # Recall tradicional sobre identificadores
    kd_ids = {idx for idx, _ in kd_pairs}
    brute_ids = {idx for idx, _ in brute_pairs}

    recall = len(kd_ids & brute_ids) / K

    # Comparación numérica de distancias de los
    # vecinos seleccionados en cada método.
    kd_distances = sorted(d for _, d in kd_pairs)
    brute_distances = sorted(d for _, d in brute_pairs)

    tie_aware = (
        len(kd_pairs) == K
        and len(brute_pairs) == K
        and np.allclose(
            kd_distances,
            brute_distances,
            atol=ABS_TOL,
            rtol=0
        )
    )

    results.append({
        "query_id": int(query_id),
        "recall_5": recall,
        "distancias_equivalentes": tie_aware
    })

df_recall = pd.DataFrame(results)

print("=== EVALUACIÓN KNN ===")
print("Consultas:", len(df_recall))
print(f"Recall@5 promedio: {df_recall['recall_5'].mean():.2%}")
print(
    "Consultas con distancias equivalentes:",
    df_recall["distancias_equivalentes"].sum()
)

print("\n=== CONSULTAS CON MENOR RECALL ===")
display(
    df_recall.sort_values("recall_5").head(10)
)


=== EVALUACIÓN KNN ===
Consultas: 100
Recall@5 promedio: 94.00%
Consultas con distancias equivalentes: 100

=== CONSULTAS CON MENOR RECALL ===


,query_id,recall_5,distancias_equivalentes
15,14316,0.6,True
25,17248,0.6,True
73,17624,0.6,True
8,3397,0.8,True
21,18987,0.8,True
22,18370,0.8,True
16,17153,0.8,True
1,19372,0.8,True
30,4939,0.8,True
52,14057,0.8,True


## 9. Evaluación de exactitud KNN considerando empates

In [10]:

import numpy as np
import pandas as pd

K = 5
ABS_TOL = 1e-7

# Recuperar suficientes candidatos para poder
# excluir siempre el propio producto.
# Para 20,621 productos, K+1 basta para excluir
# una aparición del ID consultado.
dist_kd_new, idx_kd_new = kd_tree.query(
    query_vectors,
    k=K + 1
)

dist_brute_new, idx_brute_new = brute_model.kneighbors(
    query_vectors,
    n_neighbors=K + 1
)

evaluacion = []

for i, query_position in enumerate(query_positions):

    # Excluir el propio producto de ambos resultados
    kd_neighbors = [
        (int(idx), float(dist))
        for idx, dist in zip(idx_kd_new[i], dist_kd_new[i])
        if idx != query_position
    ][:K]

    brute_neighbors = [
        (int(idx), float(dist))
        for idx, dist in zip(
            idx_brute_new[i],
            dist_brute_new[i]
        )
        if idx != query_position
    ][:K]

    complete = (
        len(kd_neighbors) == K
        and len(brute_neighbors) == K
    )

    if not complete:
        evaluacion.append({
            "query_id": int(query_position),
            "exact_match": False,
            "valid_tie_aware": False,
            "complete": False
        })
        continue

    kd_ids = {idx for idx, _ in kd_neighbors}
    brute_ids = {idx for idx, _ in brute_neighbors}

    # Umbral del quinto vecino de fuerza bruta
    kth_distance = max(
        dist for _, dist in brute_neighbors
    )

    # Todos los vecinos del KD-Tree deben estar
    # dentro del umbral exacto.
    valid_tie_aware = (
        len(kd_ids) == K
        and all(
            dist <= kth_distance + ABS_TOL
            for _, dist in kd_neighbors
        )
    )

    evaluacion.append({
        "query_id": int(query_position),
        "exact_match": kd_ids == brute_ids,
        "valid_tie_aware": valid_tie_aware,
        "complete": True
    })

df_tie_aware = pd.DataFrame(evaluacion)

print("=== EVALUACIÓN KNN CON EMPATES ===")
print("Consultas:", len(df_tie_aware))
print("Consultas completas:", df_tie_aware["complete"].sum())
print(
    "Conjuntos de vecinos idénticos:",
    df_tie_aware["exact_match"].sum()
)
print(
    "Consultas válidas considerando empates:",
    df_tie_aware["valid_tie_aware"].sum()
)

print(
    "\nExactitud considerando empates:",
    f"{df_tie_aware['valid_tie_aware'].mean():.2%}"
)

print("\n=== CONSULTAS NO VÁLIDAS ===")
display(
    df_tie_aware[
        ~df_tie_aware["valid_tie_aware"]
    ]
)


=== EVALUACIÓN KNN CON EMPATES ===
Consultas: 100
Consultas completas: 100
Conjuntos de vecinos idénticos: 73
Consultas válidas considerando empates: 100

Exactitud considerando empates: 100.00%

=== CONSULTAS NO VÁLIDAS ===


,query_id,exact_match,valid_tie_aware,complete


## 10. Validación del módulo ProductKDTree

In [11]:

import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.multidimensional_index import ProductKDTree

product_index = ProductKDTree(leaf_size=40)

product_index.fit(
    product_ids=df_features["product_id"].to_numpy(),
    features=df_features[FEATURE_COLUMNS].to_numpy()
)

# Consultar el primer producto del dataset
query_id = int(df_features.iloc[0]["product_id"])

neighbors_kd = product_index.query(
    product_id=query_id,
    k=5,
    method="kd_tree"
)

neighbors_brute = product_index.query(
    product_id=query_id,
    k=5,
    method="brute"
)

print("=== ÍNDICE MULTIDIMENSIONAL ===")
print("Productos indexados:", len(product_index.product_ids))
print("Dimensiones:", product_index.X_scaled.shape)
print("Producto consultado:", query_id)

print("\n=== VECINOS KD-TREE ===")
for neighbor in neighbors_kd:
    print(neighbor)

print("\n=== VALIDACIÓN ===")

dist_kd_mod = sorted(n["distance"] for n in neighbors_kd)
dist_brute_mod = sorted(n["distance"] for n in neighbors_brute)

assert len(neighbors_kd) == 5
assert all(n["product_id"] != query_id for n in neighbors_kd)

assert np.allclose(
    dist_kd_mod,
    dist_brute_mod,
    atol=1e-7,
    rtol=0
)

print("Producto original excluido: OK")
print("Cinco vecinos recuperados: OK")
print("Distancias equivalentes: OK")
print("\n¡Módulo ProductKDTree validado!")


=== ÍNDICE MULTIDIMENSIONAL ===
Productos indexados: 20621
Dimensiones: (20621, 4)
Producto consultado: 1001588

=== VECINOS KD-TREE ===
{'product_id': 28715802, 'distance': 0.0008209464216181739}
{'product_id': 28715852, 'distance': 0.0008209464216181739}
{'product_id': 14700810, 'distance': 0.008318011036627743}
{'product_id': 5000637, 'distance': 0.013078319658837989}
{'product_id': 3500243, 'distance': 0.015735460944673263}

=== VALIDACIÓN ===
Producto original excluido: OK
Cinco vecinos recuperados: OK
Distancias equivalentes: OK

¡Módulo ProductKDTree validado!


## 11. Validación de generación modular de características

In [12]:

from src.product_features import (
    build_product_features,
    FEATURE_COLUMNS
)

# Utilizar los mismos 100K eventos originales
features_modular = build_product_features(
    df_knn.to_dict("records")
)

# Comparar con las características del notebook
original = df_features.sort_values(
    "product_id"
).reset_index(drop=True)

modular = features_modular.sort_values(
    "product_id"
).reset_index(drop=True)

assert len(original) == len(modular)

np.testing.assert_array_equal(
    original["product_id"].to_numpy(),
    modular["product_id"].to_numpy()
)

np.testing.assert_allclose(
    original[FEATURE_COLUMNS].to_numpy(),
    modular[FEATURE_COLUMNS].to_numpy(),
    atol=1e-10,
    rtol=0
)

print("=== VALIDACIÓN DE FEATURES ===")
print("Productos:", len(modular))
print("Dimensiones:", len(FEATURE_COLUMNS))
print("Características idénticas: True")

print("\n¡Generación modular validada correctamente!")


=== VALIDACIÓN DE FEATURES ===
Productos: 20621
Dimensiones: 4
Características idénticas: True

¡Generación modular validada correctamente!


## 12. Integración del pipeline: etapas 0 a 5

In [13]:
import importlib
import src.pipeline

# Recargar el módulo después de modificar pipeline.py
importlib.reload(src.pipeline)

# Importar la nueva función
from src.pipeline import run_priority_pipeline

print("¡Función importada correctamente!")

¡Función importada correctamente!


In [14]:

import importlib
import src.pipeline

importlib.reload(src.pipeline)

from src.pipeline import run_multidimensional_pipeline

# Utilizar 1,000 eventos originales
resultado_6_etapas = run_multidimensional_pipeline(
    events=df_knn.head(1000).assign(
        event_id=lambda df: np.arange(len(df)),
        event_timestamp=lambda df: pd.to_datetime(
            df["event_time"], utc=True
        )
    ).to_dict("records"),
    batch_size=200,
    cache_capacity=100,
    max_keys=32,
    leaf_size=40
)

metrics = resultado_6_etapas["metrics"]

print("=== PIPELINE ETAPAS 0-5 ===")
print("Eventos recibidos:", metrics["received"])
print("Eventos procesados:", metrics["output"])
print("Productos en catálogo:", metrics["catalog_products"])
print("Productos indexados B+Tree:", metrics["bptree_indexed_products"])
print("Productos indexados KD-Tree:", metrics["kdtree_products"])
print("Dimensiones KD-Tree:", metrics["kdtree_dimensions"])
print("Posiciones CSR:", metrics["csr_nnz"])

# Comprobar consistencia de tamaños
assert metrics["received"] == 1000
assert metrics["output"] == 1000
assert metrics["catalog_products"] == 534
assert metrics["bptree_indexed_products"] == 534
assert metrics["kdtree_products"] == 534
assert metrics["kdtree_dimensions"] == 4
assert metrics["csr_nnz"] == 701

print("\n¡Pipeline de etapas 0-5 integrado correctamente!")


=== PIPELINE ETAPAS 0-5 ===
Eventos recibidos: 1000
Eventos procesados: 1000
Productos en catálogo: 534
Productos indexados B+Tree: 534
Productos indexados KD-Tree: 534
Dimensiones KD-Tree: 4
Posiciones CSR: 701

¡Pipeline de etapas 0-5 integrado correctamente!


In [15]:

# Obtener índice construido por el pipeline
indice_kd = resultado_6_etapas["product_kdtree"]

# Elegir un producto real del catálogo
producto_consulta = int(
    resultado_6_etapas["product_features"].iloc[0]["product_id"]
)

# Recuperar cinco vecinos
vecinos = indice_kd.query(
    product_id=producto_consulta,
    k=5,
    method="kd_tree"
)

print("=== CONSULTA KNN INTEGRADA ===")
print("Producto consultado:", producto_consulta)

for vecino in vecinos:
    print(
        f"Producto: {vecino['product_id']} | "
        f"Distancia: {vecino['distance']:.6f}"
    )

assert len(vecinos) == 5
assert all(
    v["product_id"] != producto_consulta
    for v in vecinos
)

print("\n¡Consulta KNN integrada correctamente!")


=== CONSULTA KNN INTEGRADA ===
Producto consultado: 1002524
Producto: 4804056 | Distancia: 1.378850
Producto: 5100816 | Distancia: 2.838237
Producto: 4804055 | Distancia: 13.527964
Producto: 1004856 | Distancia: 13.553446
Producto: 4802036 | Distancia: 13.622728

¡Consulta KNN integrada correctamente!
